**APPENDIX:**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier

uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
cols = ['id', 'diagnosis', 'mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness',
        'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension']

df_uci = pd.read_csv(uci_url, header=None, names=cols + [f'feat_{i}' for i in range(20)])
df_combined = pd.concat([df_uci, df_uci.copy()]).drop_duplicates().reset_index(drop=True)
df_combined = df_combined.dropna()

le = LabelEncoder()
df_combined['diagnosis'] = le.fit_transform(df_combined['diagnosis'])

X = df_combined.drop(['id', 'diagnosis'], axis=1)
y = df_combined['diagnosis']

print(f"Total Samples: {X.shape[0]}")
print(f"Total Features: {X.shape[1]}")
print(f"Feature Names: {list(X.columns[:5])}... (truncated)")
print(f"Target Classes: {list(le.classes_)} (Mapped to: {np.unique(y)})")
print("-" * 40)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)

ann_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)
ann_model.fit(X_train, y_train)

ensemble_model = VotingClassifier(estimators=[
    ('svm', svm_model), ('rf', rf_model), ('gb', gb_model)
], voting='soft')
ensemble_model.fit(X_train, y_train)

models = {
    "Support Vector Machine": svm_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "Artificial Neural Network": ann_model,
    "Ensemble Learning": ensemble_model
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    tp = cm[1, 1]
    tn = cm[0, 0]
    fp = cm[0, 1]
    fn = cm[1, 0]



    print(f"\n[MODEL: {name.upper()}]")
    print(f"Accuracy Formula: (TP+TN)/(TP+TN+FP+FN) -> ({tp}+{tn})/{len(y_test)}")
    print(f"Final Accuracy: {acc:.4f}")
    print(f"Confusion Matrix (TN, FP, FN, TP):")
    print(f"  Predicted Benign  Predicted Malignant")
    print(f"B [{tn:^15}] [{fp:^18}] (Actual)")
    print(f"M [{fn:^15}] [{tp:^18}] (Actual)")
    print("-" * 30)

Total Samples: 569
Total Features: 30
Feature Names: ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness']... (truncated)
Target Classes: ['B', 'M'] (Mapped to: [0 1])
----------------------------------------

[MODEL: SUPPORT VECTOR MACHINE]
Accuracy Formula: (TP+TN)/(TP+TN+FP+FN) -> (41+71)/114
Final Accuracy: 0.9825
Confusion Matrix (TN, FP, FN, TP):
  Predicted Benign  Predicted Malignant
B [      71       ] [        0         ] (Actual)
M [       2       ] [        41        ] (Actual)
------------------------------

[MODEL: RANDOM FOREST]
Accuracy Formula: (TP+TN)/(TP+TN+FP+FN) -> (40+70)/114
Final Accuracy: 0.9649
Confusion Matrix (TN, FP, FN, TP):
  Predicted Benign  Predicted Malignant
B [      70       ] [        1         ] (Actual)
M [       3       ] [        40        ] (Actual)
------------------------------

[MODEL: GRADIENT BOOSTING]
Accuracy Formula: (TP+TN)/(TP+TN+FP+FN) -> (40+69)/114
Final Accuracy: 0.9561
Confusion Matrix (TN, FP, FN, TP